# Database Management Systems: Week 3 - In-Depth Notes

## Week 3 Overview: From SQL Examples to Advanced SQL Features

Week 3 builds on the foundational SQL knowledge acquired in Week 2. We start with a recap of basic SQL through a series of practical examples. Then we dive into **intermediate SQL**, covering three major areas:

1. **Nested Subqueries** – Using the result of one query inside another for powerful predicate logic.
2. **Data Modification** – Inserting, deleting, and updating data with careful handling of consistency.
3. **Joins and Views** – Explicit join operations and virtual relations for data abstraction and security.
4. **Transactions, Integrity Constraints, Data Types, and Authorization** – The machinery that ensures data consistency, defines richer schemas, and controls access.

Finally, we explore **Advanced SQL**, which introduces procedural extensions (functions and procedures) and event-driven actions (**triggers**) to extend SQL beyond its declarative roots.

Throughout, the university database schema is used as a running example.

---

## Module 11: SQL Examples – A Practical Recap

This module focuses on solving representative queries using the features introduced earlier, reinforcing the practical application of SQL.

### 11.1. Basic SELECT with DISTINCT

**Example:** Find the names of buildings that contain classrooms with a capacity less than 100.

```sql
SELECT DISTINCT building
FROM classroom
WHERE capacity < 100;
```

**Explanation:**
- `FROM classroom` – We are querying the `classroom` relation.
- `WHERE capacity < 100` – This is the selection predicate. Only rows where capacity is less than 100 satisfy this condition.
- `SELECT DISTINCT building` – We want only the building names. `DISTINCT` removes duplicates because multiple classrooms in the same building may have capacity < 100.

**Without `DISTINCT`:**
```sql
SELECT building
FROM classroom
WHERE capacity < 100;
```
This returns a multiset, so if Watson appears twice (two small rooms), it appears twice in the result. By default, SQL does **not** eliminate duplicates; you must use `DISTINCT` for set semantics.

### 11.2. Cartesian Product and Join via WHERE

**Example:** List students and their department budgets where the budget is less than $100,000.

```sql
SELECT S.name, D.budget
FROM student S, department D
WHERE S.dept_name = D.dept_name
  AND D.budget < 100000;
```

**Explanation:**
- `FROM student S, department D` – This creates a Cartesian product of all students and all departments. Aliases `S` and `D` are used.
- `WHERE S.dept_name = D.dept_name` – This is the join condition that filters out meaningless combinations, keeping only rows where the student's department matches the department's name.
- `AND D.budget < 100000` – Additional condition on budget.

This is essentially a **theta join** (Cartesian product + selection).

**Using explicit renaming:**
```sql
SELECT S.name AS student_name, D.budget AS dept_budget
FROM student AS S, department AS D
WHERE S.dept_name = D.dept_name AND D.budget < 100000;
```
`AS` is optional; you can write `student S`. Column aliases rename output columns for readability.

### 11.3. Complex Predicates with AND/OR/IN

**Example:** Find names of instructors whose department is Finance or whose department is in Watson or Taylor building.

```sql
SELECT I.name
FROM instructor I, department D
WHERE I.dept_name = D.dept_name
  AND (I.dept_name = 'Finance' OR D.building IN ('Watson', 'Taylor'));
```

**Explanation:**
- The join condition `I.dept_name = D.dept_name` ensures each instructor is matched with the correct department.
- The predicate `(I.dept_name = 'Finance' OR D.building IN ('Watson', 'Taylor'))` selects instructors from Finance department, or any department housed in Watson or Taylor.
- Parentheses are crucial to group the OR correctly. Without them, due to operator precedence (AND before OR), the query would be interpreted as `(join AND I.dept_name='Finance') OR D.building IN(...)`, which is wrong.

### 11.4. String Pattern Matching with LIKE

**Example:** Find titles of courses whose course_id has three alphabetic characters followed by a hyphen.

```sql
SELECT title
FROM course
WHERE course_id LIKE '___-%';
```

**Explanation:**
- `LIKE` matches string patterns.
- `_` (underscore) matches exactly one character.
- `%` matches any sequence of zero or more characters.
- Pattern `'___-%'` means: exactly three characters (the underscores), then a hyphen, then anything (`%`).
- In the course table, course IDs like `BIO-101`, `CS-190` match; but `CS-101` (two letters) does not.

**Other pattern examples:**
- `LIKE '%dar%'` – matches any string containing "dar" anywhere.
- `LIKE 'dar%'` – starts with "dar".
- `LIKE '%dar'` – ends with "dar".
- `LIKE '_dar%'` – "dar" starts at second character.

String matching is case-sensitive in most systems.

### 11.5. ORDER BY with Multiple Attributes

**Example:** Order students by department name (ascending) and within each department by total credits (descending).

```sql
SELECT name, dept_name, tot_cred
FROM student
ORDER BY dept_name ASC, tot_cred DESC;
```

**Explanation:**
- Rows are first sorted by `dept_name` in ascending (lexicographic) order.
- Within each group of identical `dept_name`, rows are sorted by `tot_cred` in descending order.
- The order of attributes in `ORDER BY` is significant; reversing them changes the result.

### 11.6. Set Operations in Practice

**UNION Example:** Find all courses offered in Fall 2018 or Spring 2019.

```sql
(SELECT course_id FROM section WHERE semester='Fall' AND year=2018)
UNION
(SELECT course_id FROM section WHERE semester='Spring' AND year=2019);
```

- `UNION` eliminates duplicates (set semantics). Use `UNION ALL` to retain duplicates.

**INTERSECT Example:** Find names of instructors who are in Computer Science or Finance department and have salary less than 80,000.

```sql
(SELECT name FROM instructor WHERE dept_name IN ('Comp. Sci.', 'Finance'))
INTERSECT
(SELECT name FROM instructor WHERE salary < 80000);
```

**EXCEPT Example:** Find names of instructors who are in Computer Science or Finance department but whose salary is not between 70,000 and 90,000.

```sql
(SELECT name FROM instructor WHERE dept_name IN ('Comp. Sci.', 'Finance'))
EXCEPT
(SELECT name FROM instructor WHERE salary BETWEEN 70000 AND 90000);
```

**Key:** All set operations require both queries to have the same number of columns and compatible types.

### 11.7. Aggregation with GROUP BY and HAVING

**Example 1: Average capacity per building for buildings with average capacity > 25.**

```sql
SELECT building, AVG(capacity) AS avg_capacity
FROM classroom
GROUP BY building
HAVING AVG(capacity) > 25;
```

**Explanation:**
- `GROUP BY building` partitions rows into groups by building.
- `AVG(capacity)` computes the average capacity within each group.
- `HAVING` filters groups after aggregation. Only buildings with average capacity > 25 appear.

**Example 2: Total credits offered by each department.**

```sql
SELECT dept_name, SUM(credits) AS total_credits
FROM course
GROUP BY dept_name;
```

**Example 3: Count number of courses run in each building.**

```sql
SELECT building, COUNT(course_id) AS num_courses
FROM section
GROUP BY building;
```

**Important:** Attributes in `SELECT` must be either in `GROUP BY` or inside aggregate functions. `COUNT(*)` counts all rows; `COUNT(column)` ignores nulls; `COUNT(DISTINCT column)` counts distinct non-null values.

---

## Module 12: Intermediate SQL/1 – Nested Subqueries and Data Modification

### 12.1. Introduction to Subqueries

A **subquery** is a `SELECT` statement nested inside another SQL statement. Since a query always returns a relation, a subquery can be used wherever a relation or scalar value is expected.

**Placement of subqueries:**
- In the `WHERE` clause (most common) for predicates.
- In the `FROM` clause, as a derived table.
- In the `SELECT` clause, as a scalar subquery (must return a single value).

### 12.2. Subqueries in the WHERE Clause

#### 12.2.1. Set Membership: IN and NOT IN

**Example:** Find courses offered in Fall 2009 and Spring 2010 (intersection using IN).

```sql
SELECT course_id
FROM section
WHERE semester = 'Fall' AND year = 2009
  AND course_id IN (
      SELECT course_id
      FROM section
      WHERE semester = 'Spring' AND year = 2010
  );
```

**Explanation:**
- The outer query selects courses from Fall 2009.
- The subquery returns a set of course IDs offered in Spring 2010.
- `course_id IN (subquery)` checks if the Fall 2009 course also belongs to the Spring 2010 set.

Similarly, `NOT IN` can be used for set difference (EXCEPT).

**Tuple IN:** You can test multiple attributes simultaneously:

```sql
SELECT COUNT(DISTINCT ID)
FROM takes
WHERE (course_id, sec_id, semester, year) IN (
    SELECT course_id, sec_id, semester, year
    FROM teaches
    WHERE teaches.ID = '10101'
);
```
Here, the subquery returns tuples of (course_id, sec_id, semester, year). The outer query counts distinct students whose taken course sections match any tuple from that set.

#### 12.2.2. Set Comparisons: SOME and ALL

**SOME:** True if the comparison holds for at least one tuple in the subquery result.

**Example:** Find instructors whose salary is greater than that of **some** instructor in Biology.

```sql
SELECT name
FROM instructor
WHERE salary > SOME (
    SELECT salary
    FROM instructor
    WHERE dept_name = 'Biology'
);
```

**Semantics:**
- `SOME` means **exists at least one**. If there is at least one Biology salary less than the outer instructor's salary, the condition is true.
- Equivalent to using `EXISTS` with a correlated subquery.

**ALL:** True if the comparison holds for **all** tuples in the subquery result.

**Example:** Find instructors whose salary is greater than that of **all** instructors in Biology.

```sql
SELECT name
FROM instructor
WHERE salary > ALL (
    SELECT salary
    FROM instructor
    WHERE dept_name = 'Biology'
);
```

**Truth table examples for SOME and ALL:**
Given relation `R = {4, 5, 6}`:
- `5 < SOME R` → true (5 < 6).
- `5 < ALL R` → false (5 not < 4).
- `5 = SOME R` → true.
- `5 <> ALL R` → true (5 not equal to 4 or 6).

**Important:** If the subquery returns an empty set:
- `SOME` returns false (no tuple to satisfy).
- `ALL` returns true (vacuously true; all zero tuples satisfy).

#### 12.2.3. EXISTS and NOT EXISTS

`EXISTS (subquery)` returns true if the subquery returns at least one row; otherwise false. `NOT EXISTS` is the negation.

**Example:** Find all courses taught in both Fall 2009 and Spring 2010 using EXISTS.

```sql
SELECT S.course_id
FROM section S
WHERE S.semester = 'Fall' AND S.year = 2009
  AND EXISTS (
      SELECT *
      FROM section T
      WHERE T.semester = 'Spring' AND T.year = 2010
        AND T.course_id = S.course_id
  );
```

This is a **correlated subquery**: the inner query references `S.course_id` from the outer query. For each Fall 2009 row, the inner query checks if a matching Spring 2010 row exists.

**NOT EXISTS** is useful for set difference:

```sql
SELECT course_id
FROM section S
WHERE S.semester = 'Fall' AND S.year = 2009
  AND NOT EXISTS (
      SELECT *
      FROM section T
      WHERE T.semester = 'Spring' AND T.year = 2010
        AND T.course_id = S.course_id
  );
```
This returns Fall 2009 courses not offered in Spring 2010.

#### 12.2.4. UNIQUE

`UNIQUE (subquery)` returns true if the subquery returns no duplicate tuples. It tests whether all rows are distinct.

**Example:** (Illustrative) Check if there is at most one instructor in each department:

```sql
SELECT dept_name
FROM instructor
WHERE UNIQUE (
    SELECT ID
    FROM instructor I2
    WHERE I2.dept_name = instructor.dept_name
);
```

This is rarely used; `COUNT` is more common.

### 12.3. Subqueries in the FROM Clause

A subquery can be used as a derived table in the `FROM` clause. It must have an alias (though some DBMS allow omitting it if the subquery is simple).

**Example:** Find departments whose average salary is greater than 42,000.

```sql
SELECT dept_name, avg_salary
FROM (
    SELECT dept_name, AVG(salary) AS avg_salary
    FROM instructor
    GROUP BY dept_name
) AS dept_avg
WHERE avg_salary > 42000;
```

**Alternative using HAVING:**
```sql
SELECT dept_name, AVG(salary) AS avg_salary
FROM instructor
GROUP BY dept_name
HAVING AVG(salary) > 42000;
```
Both produce the same result; the derived table approach may be more readable for complex queries.

### 12.4. The WITH Clause (Common Table Expressions)

The `WITH` clause defines a temporary named relation (a Common Table Expression, CTE) that can be used in the main query.

**Example:** Find all departments with the maximum budget.

```sql
WITH max_budget (value) AS (
    SELECT MAX(budget)
    FROM department
)
SELECT dept_name
FROM department, max_budget
WHERE department.budget = max_budget.value;
```

**Advantages:**
- Improves readability by breaking complex queries into named parts.
- Can be used multiple times in the main query.
- Some DBMS support recursive WITH for hierarchical data (not covered in this course).

### 12.5. Scalar Subqueries in the SELECT Clause

A **scalar subquery** returns exactly one value (one row, one column). It can appear in the `SELECT` list.

**Example:** List all departments along with the number of instructors in each.

```sql
SELECT dept_name,
       (SELECT COUNT(*)
        FROM instructor I
        WHERE I.dept_name = D.dept_name) AS num_instructors
FROM department D;
```

**Explanation:**
- For each department row in `D`, the subquery counts instructors in that department.
- The result is a single value, so it can be used as an attribute.
- If the subquery returns more than one row, an error occurs.

### 12.6. Data Modification Statements

SQL provides three statements for modifying data: `INSERT`, `DELETE`, and `UPDATE`.

#### 12.6.1. DELETE

**Delete all tuples:**
```sql
DELETE FROM instructor;
```

**Delete with condition:**
```sql
DELETE FROM instructor
WHERE dept_name = 'Finance';
```

**Delete using a subquery:**
```sql
DELETE FROM instructor
WHERE dept_name IN (
    SELECT dept_name
    FROM department
    WHERE building = 'Watson'
);
```

**Important Pitfall:** When deleting based on an aggregate of the same table being modified, the result can be unexpected because the table changes during execution. For example:

```sql
-- Dangerous! Average changes as rows are deleted.
DELETE FROM instructor
WHERE salary < (SELECT AVG(salary) FROM instructor);
```
This may not delete the intended rows. Instead, compute the average first and store it in a variable or use a `WITH` clause.

#### 12.6.2. INSERT

**Insert a single tuple:**
```sql
INSERT INTO instructor (ID, name, dept_name, salary)
VALUES ('10102', 'Smith', 'Biology', 75000);
```
Attribute order must match the specified column list, or if no list, the table definition order.

**Insert with NULL:**
```sql
INSERT INTO student (ID, name, dept_name, tot_cred)
VALUES ('12345', 'Alice', NULL, 0);
```

**Insert from a query:**
```sql
INSERT INTO student (ID, name, dept_name, tot_cred)
SELECT ID, name, dept_name, 0
FROM instructor;
```
This inserts all instructors as students with tot_cred=0. The `SELECT` is fully evaluated before any insertion, preventing infinite loops.

#### 12.6.3. UPDATE

**Basic update:**
```sql
UPDATE instructor
SET salary = salary * 1.03
WHERE salary > 100000;
```

**Update multiple columns:**
```sql
UPDATE instructor
SET salary = salary * 1.05, dept_name = 'Math'
WHERE dept_name = 'Comp. Sci.';
```

**Update with CASE expression:**
```sql
UPDATE instructor
SET salary = CASE
    WHEN salary > 100000 THEN salary * 1.03
    ELSE salary * 1.05
END;
```
This applies a different raise based on condition in one pass, avoiding issues of multiple updates.

**Update with scalar subquery:**
```sql
UPDATE instructor
SET salary = (SELECT MAX(salary) FROM instructor)
WHERE ID = '10101';
```
Sets a specific instructor's salary to the maximum salary (scalar subquery must return one value).

---

## Module 13: Intermediate SQL/2 – Joins and Views

### 13.1. Introduction to Joins

In SQL, joins combine rows from two or more tables based on related columns. Explicit join operations in the `FROM` clause are often clearer and more efficient than implicit Cartesian products with `WHERE` conditions.

**Join Types:**
- **INNER JOIN**: Returns rows that have matching values in both tables.
- **LEFT OUTER JOIN** (or LEFT JOIN): Returns all rows from the left table, plus matched rows from the right; unmatched right rows show NULL.
- **RIGHT OUTER JOIN** (or RIGHT JOIN): Returns all rows from the right table, plus matched rows from the left; unmatched left rows show NULL.
- **FULL OUTER JOIN** (or FULL JOIN): Returns all rows from both tables; unmatched rows from either side show NULL.
- **CROSS JOIN**: Cartesian product.

**Join Conditions:**
- **NATURAL JOIN**: Automatically joins on all columns with the same name.
- **ON predicate**: Specifies the join condition.
- **USING (column_list)**: Joins on the specified common columns.

### 13.2. Example Tables: course and prereq

Consider two relations:
```
course(course_id, title, dept_name, credits)
prereq(course_id, prereq_id)
```
The `course` table contains all courses with details; `prereq` lists prerequisite courses for some courses.

**Missing information:**
- `CS-315` exists in `course` but has no entry in `prereq` (prerequisite unknown).
- `CS-347` appears in `prereq` but not in `course` (course details missing).

This allows us to illustrate how different joins handle missing matches.

### 13.3. INNER JOIN

**Example:**
```sql
SELECT *
FROM course INNER JOIN prereq ON course.course_id = prereq.course_id;
```

**Result:** Only rows where `course_id` matches in both tables are included. Rows for `CS-315` (no prereq) and `CS-347` (no course details) are excluded. This is the intersection of the two sets of course IDs.

**NATURAL INNER JOIN:**
```sql
SELECT *
FROM course NATURAL INNER JOIN prereq;
```
Since both tables have `course_id`, the natural join automatically joins on it and **removes the duplicate column**. The result has columns `course_id, title, dept_name, credits, prereq_id`.

**Note:** `INNER JOIN` is the default when you just say `JOIN`.

### 13.4. LEFT OUTER JOIN

**Example:**
```sql
SELECT *
FROM course NATURAL LEFT OUTER JOIN prereq;
```

**Result:** All rows from `course` are kept. If a course has no matching prerequisite, the `prereq_id` column is filled with NULL. Thus, `CS-315` appears with NULL prereq_id. `CS-347` is not included because it is not in the left table.

### 13.5. RIGHT OUTER JOIN

**Example:**
```sql
SELECT *
FROM course NATURAL RIGHT OUTER JOIN prereq;
```

**Result:** All rows from `prereq` are kept. If a prerequisite's course details are missing, the `title`, `dept_name`, `credits` columns are NULL. `CS-347` appears with nulls for those fields. `CS-315` (no prereq) is excluded.

### 13.6. FULL OUTER JOIN

**Example:**
```sql
SELECT *
FROM course NATURAL FULL OUTER JOIN prereq;
```

**Result:** All rows from both tables are kept. `CS-315` has NULL prereq_id; `CS-347` has NULL title/dept/credits. This preserves all information, using NULL for missing matches.

### 13.7. Join with ON and USING

- **ON**: Specifies any join predicate, not limited to equality.
  ```sql
  SELECT *
  FROM course JOIN prereq ON course.course_id = prereq.course_id AND course.dept_name = 'Biology';
  ```
- **USING**: Specifies join columns that must have the same name.
  ```sql
  SELECT *
  FROM course FULL OUTER JOIN prereq USING (course_id);
  ```
  This joins on `course_id` and removes duplicate.

**Important:** `NATURAL JOIN` can be dangerous if tables share unexpected column names. Always use `ON` or `USING` for clarity.

### 13.8. Views

A **view** is a virtual relation defined by a query. It does not store data physically; instead, its definition (query) is stored, and the view is computed on demand.

**Purpose:**
- Hide certain columns or rows from users (security).
- Simplify complex queries.
- Provide a logical abstraction independent of physical storage.

**Creating a view:**
```sql
CREATE VIEW faculty AS
SELECT ID, name, dept_name
FROM instructor;
```
The view `faculty` hides the `salary` column. Users of `faculty` see only three columns and are unaware that `salary` exists.

**Using a view:**
```sql
SELECT name
FROM faculty
WHERE dept_name = 'Biology';
```
When this query runs, the DBMS expands the view definition and executes:
```sql
SELECT name
FROM (SELECT ID, name, dept_name FROM instructor) AS faculty
WHERE dept_name = 'Biology';
```

**Views with computed attributes:**
```sql
CREATE VIEW dept_total_salary AS
SELECT dept_name, SUM(salary) AS total_salary
FROM instructor
GROUP BY dept_name;
```

### 13.9. View Expansion and Dependencies

A view can be defined using other views. When a query uses a view, the DBMS recursively replaces view names with their definitions until only base tables remain.

**Example:**
```sql
CREATE VIEW physics_fall_2009 AS
SELECT course_id
FROM section
WHERE dept_name = 'Physics' AND semester = 'Fall' AND year = 2009;

CREATE VIEW physics_fall_2009_watson AS
SELECT course_id
FROM physics_fall_2009
WHERE building = 'Watson';
```
When querying `physics_fall_2009_watson`, the system expands to include the `physics_fall_2009` definition and then the base tables.

**Dependencies:**
- View `V1` depends directly on view `V2` if `V2` is used in `V1`'s definition.
- Dependency can be transitive.
- Recursive views (where a view depends on itself) require special handling and are not covered.

### 13.10. Updates on Views

Updates (INSERT, UPDATE, DELETE) on views are often restricted because a view may not map uniquely to base tables.

**Simple view update:**
If a view is defined as `SELECT * FROM instructor WHERE dept_name='History'`, inserting a tuple with `dept_name='Biology'` would violate the view's predicate; such insert should be prevented or lead to inconsistency.

**Rules:**
- Updates on a view are generally allowed only if the view is defined from a single base table, contains the primary key, and has no aggregate, DISTINCT, GROUP BY, or computed columns.
- For complex views (joins, aggregates), updates are not directly possible. Some systems allow instead-of triggers.

**Example of problematic update:**
```sql
CREATE VIEW instructor_info AS
SELECT ID, name, building
FROM instructor, department
WHERE instructor.dept_name = department.dept_name;
```
Inserting into `instructor_info` is ambiguous because `building` is from `department`, and the corresponding `dept_name` is unknown.

### 13.11. Materialized Views

A **materialized view** is a view whose result is physically stored. It can speed up queries but requires periodic refresh to stay consistent with base tables. Materialized views are useful for expensive aggregations. This is more advanced and not covered in detail here.

---

## Module 14: Intermediate SQL/3 – Transactions, Integrity Constraints, Data Types, and Authorization

### 14.1. Transactions

A **transaction** is a logical unit of work consisting of one or more SQL statements. It must be **atomic**: either all statements are executed successfully, or none are.

**ACID Properties** (will be elaborated later):
- **Atomicity**: All-or-nothing.
- **Consistency**: Database moves from one valid state to another.
- **Isolation**: Concurrent transactions do not interfere.
- **Durability**: Once committed, changes persist.

**In SQL:**
- Transactions begin implicitly.
- End with:
  - `COMMIT`: Make changes permanent.
  - `ROLLBACK`: Undo all changes since the transaction began.

Many DBMSs auto-commit each statement by default; you can turn this off to group statements into a transaction.

**Example:**
```sql
BEGIN;
UPDATE account SET balance = balance - 100 WHERE id = 'A';
UPDATE account SET balance = balance + 100 WHERE id = 'B';
COMMIT; -- or ROLLBACK on error
```

### 14.2. Integrity Constraints

Integrity constraints are rules that ensure data validity and consistency.

#### 14.2.1. Single-Relation Constraints

- **NOT NULL**: Attribute cannot be null.
  ```sql
  CREATE TABLE instructor (
      name VARCHAR(20) NOT NULL
  );
  ```
- **UNIQUE**: A set of attributes must have unique values across tuples. Unlike primary key, UNIQUE allows NULL (and in many DBMS multiple NULLs are allowed).
  ```sql
  CREATE TABLE instructor (
      ID CHAR(5) UNIQUE
  );
  ```
- **PRIMARY KEY**: Combines NOT NULL and UNIQUE. One per table.
- **CHECK**: A predicate that each tuple must satisfy.
  ```sql
  CREATE TABLE section (
      semester VARCHAR(6),
      CHECK (semester IN ('Fall', 'Winter', 'Spring', 'Summer'))
  );
  ```

#### 14.2.2. Referential Integrity (Foreign Keys)

A foreign key establishes a link between two tables. The referencing attribute(s) must match the primary key of the referenced table.

**Example:**
```sql
CREATE TABLE instructor (
    ...
    FOREIGN KEY (dept_name) REFERENCES department(dept_name)
);
```

**Referential Actions on Delete/Update:**
- `CASCADE`: If referenced tuple is deleted/updated, referencing tuples are also deleted/updated.
  ```sql
  FOREIGN KEY (dept_name) REFERENCES department(dept_name) ON DELETE CASCADE
  ```
- `SET NULL`: Set foreign key to NULL.
- `SET DEFAULT`: Set to default value.
- `NO ACTION` (default): Prohibit operation if it would violate constraint.

**Self-referencing foreign keys:**
```sql
CREATE TABLE person (
    ID INT PRIMARY KEY,
    mother_ID INT,
    father_ID INT,
    FOREIGN KEY (mother_ID) REFERENCES person(ID),
    FOREIGN KEY (father_ID) REFERENCES person(ID)
);
```
Inserting a person whose parents are not yet in the table requires careful ordering or deferring constraints.

#### 14.2.3. Deferring Constraint Checks

Some systems allow deferring constraint checks until commit. This helps with cyclic dependencies.

### 14.3. SQL Data Types and Schemas

#### 14.3.1. Date, Time, Timestamp, Interval

- **DATE**: Stores year, month, day.
- **TIME**: Stores hour, minute, second.
- **TIMESTAMP**: DATE + TIME.
- **INTERVAL**: A duration, e.g., INTERVAL '1' DAY.
- Operations: Subtract dates to get an interval; add interval to date.

**Example:**
```sql
SELECT DATE '2023-01-01' + INTERVAL '1' MONTH;
```
Returns 2023-02-01.

#### 14.3.2. Indexes

An **index** is a separate data structure that speeds up data retrieval. It is created on one or more columns of a table.

**Syntax:**
```sql
CREATE INDEX student_id_index ON student(ID);
```

**Benefits:** Faster search/join on indexed columns.
**Cost:** Extra storage and slower inserts/updates.

#### 14.3.3. User-Defined Types and Domains

**User-Defined Type (UDT):** An alias for a built-in type, improving readability.
```sql
CREATE TYPE Dollars AS NUMERIC(12,2);
```
Then use `Dollars` as a column type.

**Domain:** A UDT with additional constraints.
```sql
CREATE DOMAIN PersonName AS VARCHAR(20) NOT NULL;
CREATE DOMAIN DegreeLevel AS VARCHAR(10)
    CHECK (VALUE IN ('Bachelors', 'Masters', 'Doctorate'));
```
Domains can be reused across tables, enforcing consistent constraints.

#### 14.3.4. Large Object Types (BLOB, CLOB)

- **BLOB** (Binary Large Object): Stores binary data like images, videos.
- **CLOB** (Character Large Object): Stores large text.
- Usually stored outside the DBMS file system, with a pointer in the table.

### 14.4. Authorization

Authorization controls what users can do with data. Privileges are granted to users or roles.

**Privilege types:**
- **Read**: SELECT
- **Insert**: INSERT
- **Update**: UPDATE
- **Delete**: DELETE
- **Schema privileges**: CREATE, ALTER, DROP, REFERENCES, INDEX

#### 14.4.1. GRANT

**Syntax:**
```sql
GRANT <privilege_list> ON <relation> TO <user_list>;
```

**Examples:**
```sql
GRANT SELECT ON instructor TO Amit;
GRANT INSERT, DELETE ON student TO Amit, Rohit;
GRANT ALL PRIVILEGES ON department TO dean;
```

- `PUBLIC` keyword grants to all users.
- Can grant on views as well, providing column-level security without granting on base tables.

#### 14.4.2. REVOKE

Removes previously granted privileges.

```sql
REVOKE SELECT ON instructor FROM Amit;
REVOKE ALL PRIVILEGES ON student FROM PUBLIC;
```

#### 14.4.3. Roles

A **role** is a named set of privileges. It simplifies management.

**Example:**
```sql
CREATE ROLE instructor;
GRANT SELECT ON takes TO instructor;
GRANT instructor TO Amit;
```
Amit now has SELECT on takes.

Roles can be granted to other roles, creating a hierarchy:
```sql
CREATE ROLE dean;
GRANT instructor TO dean;  -- dean inherits instructor privileges
GRANT dean TO Satoshi;
```

#### 14.4.4. Views for Authorization

Views can be used to grant access to a subset of data without granting access to the underlying table.

```sql
CREATE VIEW geo_instructor AS
SELECT ID, name, dept_name
FROM instructor
WHERE dept_name = 'Geology';

GRANT SELECT ON geo_instructor TO geo_staff;
```
`geo_staff` can query `geo_instructor` but does not have permission on the entire `instructor` table.

#### 14.4.5. Grant Option and References

- **WITH GRANT OPTION**: Allows the grantee to grant the same privilege to others.
  ```sql
  GRANT SELECT ON department TO Amit WITH GRANT OPTION;
  ```
- **REFERENCES**: Needed to create foreign keys referencing a table.
  ```sql
  GRANT REFERENCES (dept_name) ON department TO Amit;
  ```

Revoking privileges can cascade if `CASCADE` is specified; default may be `RESTRICT`.

---

## Module 15: Advanced SQL – Functions, Procedures, and Triggers

### 15.1. Introduction

SQL is primarily declarative, but SQL:1999 added procedural extensions to allow more complex application logic within the database. This reduces network overhead and encapsulates business rules.

**Two models of integrating procedural code:**
1. **External language routines**: Functions written in C, Java, etc., called from SQL.
2. **Embedded SQL**: SQL statements embedded in a host language program.

In this module, we focus on **SQL functions and procedures** (written in SQL or external languages) and **triggers**.

### 15.2. SQL Functions

A **function** is a named block of code that returns a value. It can take parameters and is used like a scalar expression.

**Syntax:**
```sql
CREATE FUNCTION dept_count (dept_name VARCHAR(20))
RETURNS INTEGER
BEGIN
    DECLARE d_count INTEGER;
    SELECT COUNT(*) INTO d_count
    FROM instructor
    WHERE instructor.dept_name = dept_name;
    RETURN d_count;
END;
```

**Explanation:**
- `CREATE FUNCTION dept_count (dept_name VARCHAR(20))` defines function name and parameters.
- `RETURNS INTEGER` specifies return type.
- `BEGIN ... END` encloses the body.
- `DECLARE d_count INTEGER;` local variable.
- `SELECT COUNT(*) INTO d_count` assigns query result to variable.
- `RETURN d_count;` returns the value.

**Usage:**
```sql
SELECT dept_name, budget
FROM department
WHERE dept_count(dept_name) > 12;
```
The function is invoked for each row, acting as a parameterized query.

### 15.3. Table Functions

A **table function** returns a table (relation). This is useful for returning multiple rows.

**Syntax:**
```sql
CREATE FUNCTION instructor_of (dept_name VARCHAR(20))
RETURNS TABLE (
    ID VARCHAR(5),
    name VARCHAR(20),
    dept_name VARCHAR(20)
)
RETURN TABLE (
    SELECT ID, name, dept_name
    FROM instructor
    WHERE instructor.dept_name = instructor_of.dept_name
);
```

**Usage:**
```sql
SELECT *
FROM TABLE (instructor_of('Music')) AS music_instructors;
```
The `TABLE` keyword indicates the function returns a table.

### 15.4. SQL Procedures

A **procedure** is similar to a function but does not return a value. It can have IN, OUT, and INOUT parameters.

**Syntax:**
```sql
CREATE PROCEDURE dept_count_proc (
    IN dept_name VARCHAR(20),
    OUT d_count INTEGER
)
BEGIN
    SELECT COUNT(*) INTO d_count
    FROM instructor
    WHERE instructor.dept_name = dept_name_proc.dept_name;
END;
```

**Calling:**
```sql
DECLARE d_count INTEGER;
CALL dept_count_proc('Physics', d_count);
```
Procedures are invoked with `CALL`, not embedded in expressions.

**Overloading:** SQL allows overloading of functions/procedures (same name, different parameters).

### 15.5. Procedural Constructs in SQL

SQL provides control-flow constructs similar to other procedural languages.

#### 15.5.1. Compound Statements

```sql
BEGIN
    -- statements
END
```
Allows grouping multiple statements.

#### 15.5.2. While Loop

```sql
WHILE boolean_expression DO
    statement_list;
END WHILE;
```
Checks condition before each iteration.

#### 15.5.3. Repeat Loop

```sql
REPEAT
    statement_list;
UNTIL boolean_expression
END REPEAT;
```
Executes at least once.

#### 15.5.4. For Loop

Iterates over query results:
```sql
DECLARE n INTEGER DEFAULT 0;
FOR r AS SELECT budget FROM department
DO
    SET n = n + r.budget;
END FOR;
```
This is similar to a cursor-based loop.

#### 15.5.5. If-Then-Else

```sql
IF boolean_expression THEN
    statements;
ELSEIF boolean_expression THEN
    statements;
ELSE
    statements;
END IF;
```

#### 15.5.6. Case Statement

Two forms:
- **Simple case**:
  ```sql
  CASE variable
      WHEN value1 THEN statements1;
      WHEN value2 THEN statements2;
      ELSE statements3;
  END CASE;
  ```
- **Searched case**:
  ```sql
  CASE
      WHEN condition1 THEN statements1;
      WHEN condition2 THEN statements2;
      ELSE statements3;
  END CASE;
  ```

#### 15.5.7. Exception Handling

You can define and raise exceptions, with handlers.

**Example:**
```sql
DECLARE out_of_classroom_seats CONDITION;
DECLARE EXIT HANDLER FOR out_of_classroom_seats
BEGIN
    -- action
END;
...
IF seat_count > capacity THEN
    SIGNAL out_of_classroom_seats;
END IF;
```

### 15.6. External Language Routines

Functions/procedures can be written in external languages like C or Java. This is useful for complex algorithms or specialized data types.

**Example:**
```sql
CREATE PROCEDURE my_proc (IN param1 INT)
LANGUAGE C
EXTERNAL NAME '/path/to/function';
```

**Benefits:** Efficiency, access to libraries.
**Risks:** Security, potential corruption if unsafe languages are used. Use safe languages (Java, etc.) or sandboxing.

### 15.7. Triggers

A **trigger** is a special kind of stored procedure that automatically executes when a specified event occurs on a table. Events are `INSERT`, `DELETE`, `UPDATE`.

**Purpose:**
- Enforce complex integrity constraints.
- Maintain derived data (e.g., update total credits).
- Audit logging.
- Enforce business rules.

**Trigger Components:**
- **Event**: Which DML operation.
- **Timing**: `BEFORE` or `AFTER`.
- **Level**: `FOR EACH ROW` or `FOR EACH STATEMENT`.
- **Action**: The code to execute.

**Example: BEFORE UPDATE Trigger**
```sql
CREATE TRIGGER setnull_trigger
BEFORE UPDATE ON takes
REFERENCING NEW ROW AS nrow
FOR EACH ROW
WHEN (nrow.grade = '')
BEGIN ATOMIC
    SET nrow.grade = NULL;
END;
```
This trigger runs before an UPDATE on `takes`. If the new grade is an empty string, it is replaced with NULL.

**Example: AFTER UPDATE Trigger for total credits**
```sql
CREATE TRIGGER credits_earned
AFTER UPDATE OF takes ON (grade)
REFERENCING NEW ROW AS nrow
REFERENCING OLD ROW AS orow
FOR EACH ROW
WHEN (nrow.grade <> 'F' AND nrow.grade IS NOT NULL
      AND (orow.grade = 'F' OR orow.grade IS NULL))
BEGIN ATOMIC
    UPDATE student
    SET tot_cred = tot_cred + (
        SELECT credits
        FROM course
        WHERE course.course_id = nrow.course_id
    )
    WHERE student.ID = nrow.ID;
END;
```
This trigger adds credits when a student receives a passing grade.

**Statement-level vs Row-level:**
- `FOR EACH ROW`: fires once per affected row.
- `FOR EACH STATEMENT`: fires once per SQL statement, regardless of row count.

**Before vs After:**
- `BEFORE`: Can modify new values before they are written.
- `AFTER`: Used to perform actions after data has changed, e.g., update other tables.

**Referencing old/new values:**
- `REFERENCING OLD ROW AS orow` and `NEW ROW AS nrow` for row-level.
- `REFERENCING OLD TABLE AS otab` and `NEW TABLE AS ntab` for statement-level.

### 15.8. When to Use Triggers (Best Practices)

**Good uses:**
- Logging changes (audit trail).
- Enforcing complex referential integrity.
- Deriving computed values.
- Simple validation.

**Bad uses / pitfalls:**
- Too many triggers lead to performance degradation.
- Complex trigger code is hard to debug.
- Recursive triggers should be avoided (usually disabled by default).
- Triggers that call other triggers (chain) can lead to unexpected behavior.
- Using functions/procedures/views inside triggers can be expensive.
- Avoid iteration in triggers; use set-based logic.

**Golden rule:** Keep triggers simple, short, and few. They are powerful but dangerous if misused.

---

## Summary

In Week 3, we have covered:

- **SQL Examples**: Practical use of SELECT, DISTINCT, joins via WHERE, IN, LIKE, ORDER BY, set operations, and aggregation.
- **Intermediate SQL/1**: Nested subqueries with IN, SOME/ALL, EXISTS, UNIQUE, scalar subqueries, WITH clause, and data modification statements (INSERT, DELETE, UPDATE) with case expressions.
- **Intermediate SQL/2**: Explicit joins (inner, left/right/full outer, natural, ON, USING) and views (virtual relations, view expansion, update restrictions, materialized views).
- **Intermediate SQL/3**: Transactions (ACID, COMMIT/ROLLBACK), integrity constraints (NOT NULL, UNIQUE, CHECK, foreign keys with cascading), additional data types (date/time, indexes, domains, large objects), and authorization (GRANT, REVOKE, roles, views for security).
- **Advanced SQL**: Functions, table functions, procedures, procedural constructs (loops, conditionals, exceptions), external language routines, and triggers (events, timing, row/statement level, best practices).

These concepts provide a robust foundation for building and managing relational database applications. In subsequent weeks, we will delve deeper into relational database design, normalization, and the internal workings of a DBMS.